# Підключення бібліотек

In [44]:
import pandas as pd
import numpy as np

In [45]:
interactions = pd.read_parquet("filtered_interactions.parquet")
items = pd.read_parquet("filtered_items.parquet")

print(interactions.shape)
print(items.shape)

(56163348, 3)
(7189, 5)


In [46]:
user_genres = interactions.merge(
    items[["anime_id", "Genres"]],
    on="anime_id",
    how="left"
)

print(user_genres.shape)
display(user_genres.head())

(56163348, 4)


,user_id,anime_id,rating,Genres
0,0,430,9,"Military, Comedy, Historical, Drama, Fantasy, ..."
1,0,1004,5,"Drama, Psychological, Romance, Slice of Life"
2,0,3010,7,"Adventure, Historical, Shounen"
3,0,570,7,"Military, Police, Psychological, Drama, Romance"
4,0,2762,9,"Action, Adventure, Comedy, Romance, School, Sh..."


In [48]:
from sklearn.preprocessing import MultiLabelBinarizer

items_features = items.copy()

items_features["genre_list"] = (
    items_features["Genres"]
    .fillna("")
    .apply(
        lambda x: [g.strip() for g in x.split(",") if g.strip()]
    )
)

mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(
    items_features["genre_list"]
)

genre_features = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=items_features.index
)

print(genre_features.shape)
display(genre_features)

(7189, 42)


,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,...,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi
0,1,1,0,1,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,1,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1,1,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7184,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7185,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7186,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7187,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
genre_features.insert(
    0,
    "anime_id",
    items_features["anime_id"].values
)

In [50]:
print(genre_features.shape)
print(genre_features.columns.tolist())
display(genre_features.head())

(7189, 43)
['anime_id', 'Action', 'Adventure', 'Cars', 'Comedy', 'Dementia', 'Demons', 'Drama', 'Ecchi', 'Fantasy', 'Game', 'Harem', 'Historical', 'Horror', 'Josei', 'Kids', 'Magic', 'Martial Arts', 'Mecha', 'Military', 'Music', 'Mystery', 'Parody', 'Police', 'Psychological', 'Romance', 'Samurai', 'School', 'Sci-Fi', 'Seinen', 'Shoujo', 'Shoujo Ai', 'Shounen', 'Shounen Ai', 'Slice of Life', 'Space', 'Sports', 'Super Power', 'Supernatural', 'Thriller', 'Unknown', 'Vampire', 'Yaoi']


,anime_id,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,...,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi
0,1,1,1,0,1,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
1,5,1,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
2,6,1,1,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,7,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,8,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0


In [51]:
print(genre_features["anime_id"].nunique())
print(genre_features["anime_id"].is_unique)

print(genre_features.iloc[:5, :8])

7189
True
   anime_id  Action  Adventure  Cars  Comedy  Dementia  Demons  Drama
0         1       1          1     0       1         0       0      1
1         5       1          0     0       0         0       0      1
2         6       1          1     0       1         0       0      1
3         7       1          0     0       0         0       0      1
4         8       0          1     0       0         0       0      0


In [52]:
valid_item_ids = genre_features["anime_id"].unique()

filtered_interactions = interactions[
    interactions["anime_id"].isin(valid_item_ids)
].copy()

print("Interactions:", filtered_interactions.shape)
print("Users:", filtered_interactions["user_id"].nunique())
print("Anime:", filtered_interactions["anime_id"].nunique())

Interactions: (46766934, 3)
Users: 264515
Anime: 7189


In [53]:
print(
    "Missing anime IDs:",
    (
        ~filtered_interactions["anime_id"].isin(
            genre_features["anime_id"]
        )
    ).sum()
)

Missing anime IDs: 0


In [54]:
filtered_items = items[
    items["anime_id"].isin(
        filtered_interactions["anime_id"]
    )
].copy()

print("Items:", filtered_items.shape)

Items: (7189, 5)


In [55]:
from scipy.sparse import csr_matrix

# --------------------------------------------------
# 1. Створюємо відповідність anime_id -> номер рядка
# --------------------------------------------------

item_ids = genre_features["anime_id"].to_numpy()

anime_to_idx = pd.Series(
    np.arange(len(item_ids)),
    index=item_ids
)

# --------------------------------------------------
# 2. Перетворюємо user_id у компактні індекси
# --------------------------------------------------

user_ids = filtered_interactions["user_id"].unique()

user_to_idx = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

# --------------------------------------------------
# 3. Індекси для sparse user-item matrix
# --------------------------------------------------

rows = filtered_interactions["user_id"].map(user_to_idx).to_numpy()
cols = filtered_interactions["anime_id"].map(anime_to_idx).to_numpy()

ratings_values = filtered_interactions["rating"].to_numpy(dtype=np.float32)

# --------------------------------------------------
# 4. User × Anime rating matrix
# --------------------------------------------------

R = csr_matrix(
    (
        ratings_values,
        (rows, cols)
    ),
    shape=(len(user_ids), len(item_ids))
)

print("Rating matrix:", R.shape)
print("Non-zero ratings:", R.nnz)

Rating matrix: (264515, 7189)
Non-zero ratings: 46766934


In [56]:
genre_columns = [
    col for col in genre_features.columns
    if col != "anime_id"
]

G = csr_matrix(
    genre_features[genre_columns].to_numpy(
        dtype=np.float32
    )
)

print("Genre matrix:", G.shape)

Genre matrix: (7189, 42)


In [57]:
# Сума рейтингів користувача за кожним жанром
rating_sums = R @ G

# Кількість тайтлів користувача в кожному жанрі
rating_counts = (R != 0).astype(np.float32) @ G

print(rating_sums.shape)
print(rating_counts.shape)

(264515, 42)
(264515, 42)


In [58]:
user_genre_mean = np.divide(
    rating_sums.toarray(),
    rating_counts.toarray(),
    out=np.zeros_like(rating_sums.toarray()),
    where=rating_counts.toarray() != 0
)

In [59]:
user_genre_profile = pd.DataFrame(
    user_genre_mean,
    columns=genre_columns
)

user_genre_profile.insert(
    0,
    "user_id",
    user_ids
)

In [60]:
print(user_genre_profile.shape)
display(user_genre_profile.head())

(264515, 43)


,user_id,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,...,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi
0,0,7.700000,7.444445,0.0,7.750000,0.0,0.000000,7.200000,4.000000,7.818182,...,6.0,7.000000,0.0,6.000000,6.666667,7.571429,0.000000,0.0,6.50,0.0
1,1,7.933333,7.647059,0.0,7.764706,0.0,8.000000,8.523809,7.500000,7.692307,...,0.0,8.000000,0.0,7.833333,8.500000,7.928571,9.333333,0.0,8.00,0.0
2,2,8.600000,9.142858,0.0,8.533334,0.0,8.500000,8.727273,8.000000,9.333333,...,0.0,8.500000,0.0,9.000000,9.111111,8.357142,8.750000,0.0,9.00,0.0
3,3,7.541285,7.782609,8.0,7.470930,8.0,7.636364,7.949152,7.064935,7.550000,...,0.0,7.754386,9.0,8.000000,7.000000,7.604651,8.000000,0.0,7.25,0.0
4,4,7.531250,7.769231,0.0,7.685185,0.0,6.666667,7.724638,6.500000,7.564103,...,7.0,7.821429,0.0,7.000000,7.888889,7.676471,8.333333,0.0,8.50,0.0


In [61]:
genre_columns = user_genre_profile.columns.drop('user_id')

user_genre_profile[genre_columns] = (
    user_genre_profile[genre_columns].replace(0, np.nan)
)
display(user_genre_profile)

,user_id,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,...,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi
0,0,7.700000,7.444445,NaN,7.750000,NaN,NaN,7.200000,4.000000,7.818182,...,6.0,7.000000,NaN,6.000000,6.666667,7.571429,NaN,NaN,6.500000,NaN
1,1,7.933333,7.647059,NaN,7.764706,NaN,8.000000,8.523809,7.500000,7.692307,...,NaN,8.000000,NaN,7.833333,8.500000,7.928571,9.333333,NaN,8.000000,NaN
2,2,8.600000,9.142858,NaN,8.533334,NaN,8.500000,8.727273,8.000000,9.333333,...,NaN,8.500000,NaN,9.000000,9.111111,8.357142,8.750000,NaN,9.000000,NaN
3,3,7.541285,7.782609,8.0,7.470930,8.000000,7.636364,7.949152,7.064935,7.550000,...,NaN,7.754386,9.0,8.000000,7.000000,7.604651,8.000000,NaN,7.250000,NaN
4,4,7.531250,7.769231,NaN,7.685185,NaN,6.666667,7.724638,6.500000,7.564103,...,7.0,7.821429,NaN,7.000000,7.888889,7.676471,8.333333,NaN,8.500000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264510,353398,6.271357,6.371795,NaN,6.440000,7.000000,5.850000,6.701220,5.601942,6.278688,...,NaN,6.733945,7.0,6.950000,6.325582,6.229508,6.666667,NaN,6.307693,NaN
264511,353400,8.592592,8.928572,NaN,8.500000,NaN,8.000000,8.777778,8.272727,8.133333,...,NaN,8.294118,NaN,8.000000,9.111111,8.650000,8.000000,NaN,8.333333,NaN
264512,353401,7.452381,8.125000,NaN,7.800000,8.666667,7.000000,7.142857,8.000000,7.750000,...,NaN,8.000000,NaN,NaN,8.000000,7.740741,8.600000,NaN,7.000000,NaN
264513,353403,8.709678,9.166667,NaN,8.750000,NaN,NaN,9.157895,8.000000,8.875000,...,NaN,9.500000,9.0,9.000000,9.000000,8.714286,9.500000,NaN,9.000000,NaN


In [67]:
sample_interactions = filtered_interactions.sample(
    n=500_000,
    random_state=42
)

print(sample_interactions.shape)

(500000, 3)


In [70]:
train_data = sample_interactions.merge(
    user_genre_profile,
    on="user_id",
    how="inner"
)

print(train_data.shape)

(500000, 45)


In [71]:
item_genres = genre_features.copy()

item_genres = item_genres.rename(
    columns={
        col: f"item_{col}"
        for col in item_genres.columns
        if col != "anime_id"
    }
)

In [72]:
train_data = train_data.merge(
    item_genres,
    on="anime_id",
    how="inner"
)

print(train_data.shape)

(500000, 87)


In [73]:
genre_columns = [
    col for col in genre_features.columns
    if col != "anime_id"
]

user_feature_columns = genre_columns

item_feature_columns = [
    f"item_{genre}"
    for genre in genre_columns
]

feature_columns = (
    user_feature_columns +
    item_feature_columns
)

X = train_data[feature_columns].copy()
y = train_data["rating"].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (500000, 84)
y: (500000,)


In [74]:
X = X.fillna(X.mean())

In [75]:
print("NaN:", X.isna().sum().sum())

NaN: 0


In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer

items_text = filtered_items[["anime_id", "sypnopsis"]].copy()

items_text["sypnopsis"] = items_text["sypnopsis"].fillna("")

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=3,
    max_df=0.95,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(items_text["sypnopsis"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (7189, 5000)


In [86]:
anime_to_tfidf_idx = pd.Series(
    items_text.index,
    index=items_text["anime_id"]
)

In [87]:
items_text = filtered_items[["anime_id", "sypnopsis"]].copy().reset_index(drop=True)

items_text["sypnopsis"] = items_text["sypnopsis"].fillna("")

tfidf_matrix = tfidf.fit_transform(items_text["sypnopsis"])

anime_to_idx = pd.Series(
    items_text.index,
    index=items_text["anime_id"]
)

In [88]:
tfidf_indices = sample_interactions["anime_id"].map(anime_to_idx)

print("Missing:", tfidf_indices.isna().sum())

Missing: 0


In [89]:
tfidf_indices = tfidf_indices.to_numpy(dtype=np.int32)

X_text = tfidf_matrix[tfidf_indices]

print("X_text:", X_text.shape)

X_text: (500000, 5000)


In [90]:
from scipy.sparse import csr_matrix, hstack

X_sparse = hstack([
    csr_matrix(X.to_numpy(dtype=np.float32)),
    X_text
])

print("Final X:", X_sparse.shape)

Final X: (500000, 5084)


In [93]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_sparse,
    y,
    test_size=0.2,
    random_state=42
)

In [94]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

In [95]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.939177736495047
RMSE: 1.2406929011478602
R²: 0.44721439308658495


In [129]:
import joblib

joblib.dump(model, 'models/linear_regression_model.joblib')
joblib.dump(tfidf, 'models/tfidf_vectorizer.joblib')
joblib.dump(mlb, 'models/multilabel_binarizer.joblib')

['models/multilabel_binarizer.joblib']

In [130]:
from scipy import sparse
import os

os.makedirs("data/training_splits", exist_ok=True)

# Зберігаємо розріджені матриці X (фічі)
sparse.save_npz("data/training_splits/X_train.npz", X_train)
sparse.save_npz("data/training_splits/X_test.npz", X_test)

# Зберігаємо вектори y (таргети)
np.save("data/training_splits/y_train.npy", y_train.to_numpy())
np.save("data/training_splits/y_test.npy", y_test.to_numpy())

print("Тренувальні та тестові дані успішно збережено!")
# Збереження

Тренувальні та тестові дані успішно збережено!


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/X_sparse.npz'

In [131]:
sparse.save_npz("data/processed/X_sparse.npz", X_sparse)
np.save("data/processed/y.npy", y.to_numpy())

# test

In [112]:
test_history = filtered_items[
    filtered_items["Name"].isin([
        "Fullmetal Alchemist: The Conqueror of Shamballa",
        "Jin-Rou",
        "Kanojo to Kanojo no Neko"
    ])
].copy()
display(test_history[["anime_id", "Name", "Genres", "sypnopsis"]])

test_history["rating"] = [9, 9, 5]

display(test_history[[
    "anime_id",
    "Name",
    "rating",
    "Genres"
]])

,anime_id,Name,Genres,sypnopsis
390,430,Fullmetal Alchemist: The Conqueror of Shamballa,"Military, Comedy, Historical, Drama, Fantasy, ...","In desperation, Edward Elric sacrificed his bo..."
522,570,Jin-Rou,"Military, Police, Psychological, Drama, Romance","In an alternate history, following World War I..."
879,1004,Kanojo to Kanojo no Neko,"Drama, Psychological, Romance, Slice of Life",It was on a rainy spring day that Chobi became...


,anime_id,Name,rating,Genres
390,430,Fullmetal Alchemist: The Conqueror of Shamballa,9,"Military, Comedy, Historical, Drama, Fantasy, ..."
522,570,Jin-Rou,9,"Military, Police, Psychological, Drama, Romance"
879,1004,Kanojo to Kanojo no Neko,5,"Drama, Psychological, Romance, Slice of Life"


In [113]:
test_history_genres = test_history.merge(
    genre_features,
    on="anime_id",
    how="left"
)

In [114]:
genre_columns = [
    col for col in genre_features.columns
    if col != "anime_id"
]

user_genre_values = {}

for genre in genre_columns:
    mask = test_history_genres[genre] == 1

    if mask.any():
        user_genre_values[genre] = (
            test_history_genres.loc[mask, "rating"].mean()
        )
    else:
        user_genre_values[genre] = np.nan

In [115]:
test_user_profile = pd.DataFrame(
    [user_genre_values]
)

display(test_user_profile)

,Action,Adventure,Cars,Comedy,Dementia,Demons,Drama,Ecchi,Fantasy,Game,...,Shounen Ai,Slice of Life,Space,Sports,Super Power,Supernatural,Thriller,Unknown,Vampire,Yaoi
0,NaN,NaN,NaN,9.0,NaN,NaN,7.666667,NaN,9.0,NaN,...,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
seen_ids = test_history["anime_id"]

candidate = filtered_items[
    ~filtered_items["anime_id"].isin(seen_ids)
].sample(1, random_state=42)

display(candidate)

,anime_id,Name,Score,Genres,sypnopsis
1318,1528,Bishoujo Yuugekitai Battle Skipper,5.25,"Sci-Fi, Comedy, Mecha","Despite the title, this is not a feature but t..."


In [117]:
candidate_genres = genre_features[
    genre_features["anime_id"] == candidate.iloc[0]["anime_id"]
].iloc[0]

In [118]:
user_features = []

for genre in genre_columns:
    user_features.append(
        test_user_profile.iloc[0][genre]
    )

item_features = []

for genre in genre_columns:
    item_features.append(
        candidate_genres[genre]
    )

X_new = pd.DataFrame(
    [user_features + item_features],
    columns=feature_columns
)
print(X_new.shape)

(1, 84)


In [119]:
candidate_anime_id = candidate.iloc[0]["anime_id"]

tfidf_idx = anime_to_idx[candidate_anime_id]

candidate_tfidf = tfidf_matrix[tfidf_idx]
print(candidate_tfidf.shape)

(1, 5000)


In [126]:
from scipy.sparse import csr_matrix, hstack

X_new_sparse = hstack([
    csr_matrix(X_new.to_numpy(dtype=np.float32)),
    candidate_tfidf
])

print(X_new_sparse.shape)

(1, 5084)


In [127]:
predicted_rating = model.predict(X_new_sparse)[0]

print(
    f"Predicted rating: {predicted_rating:.2f}"
)

Predicted rating: 5.17
